# Holo Jupyter — content-addressed, sub-millisecond CPU (WASM) + GPU (WebGPU)

CPython runs as **WebAssembly** on your CPU; `holo_gpu` dispatches **WGSL** to your real **GPU** via WebGPU. Operands carry their content address (κ) so repeats and chains run **well below 1 ms**. No server, no install. Run all cells.

In [ ]:
# One-time, fully offline: install the interactive stack from the bundled wheel index (no network).
import piplite
await piplite.install(['ipywidgets', 'ipympl', 'plotly', 'pandas', 'nbformat'])
print('interactive stack ready (offline)')

In [ ]:
import numpy as np, holo_gpu, time
await holo_gpu.warmup()              # acquire the GPU device once (reused for every dispatch)
print('WebGPU available:', holo_gpu.available())
await holo_gpu.adapter_info()

## 1 · Square every element on the GPU
The kernel binds one `read_write` storage buffer at `@group(0) @binding(0)`.

In [ ]:
wgsl_sq = r"""
@group(0) @binding(0) var<storage, read_write> data: array<f32>;
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let i = gid.x;
    if (i < arrayLength(&data)) { data[i] = data[i] * data[i]; }
}"""
x = np.arange(16, dtype=np.float32)
y = await holo_gpu.run(wgsl_sq, x)
print('x =', x)
print('y =', np.asarray(y))
assert np.allclose(np.asarray(y), x * x)
print('GPU result matches numpy')

## 2 · Sub-millisecond by content address (the O(1) win)
`holo_gpu.address(arr)` computes the operand's κ **once**. After that, a repeat of the same kernel on the same content is a hash-map lookup — **microseconds** (Law L5: same content ⊕ same kernel ⇒ same result). A single *cold* GPU readback can't beat WebGPU's ~few-ms sync floor; the CPU and the O(1) cache can.

In [ ]:
def us(t): return f'{t*1e6:9.1f} us'
wgsl = r"""
@group(0) @binding(0) var<storage, read_write> d: array<f32>;
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) g: vec3<u32>){ let i=g.x; if(i<arrayLength(&d)){ d[i]=sqrt(d[i])*1.5+2.0; } }
"""
xk = holo_gpu.address((np.arange(1<<20, dtype=np.float32) % 1000))   # 1,048,576 floats, addressed once
print('operand:', xk)
t=time.perf_counter(); r1=await holo_gpu.run(wgsl, xk);  t_cold=time.perf_counter()-t
t=time.perf_counter(); r2=await holo_gpu.run(wgsl, xk);  t_hit =time.perf_counter()-t
t=time.perf_counter(); cpu=xk.data*1.5+2.0;              t_cpu =time.perf_counter()-t
print('GPU cold (dispatch+readback):', us(t_cold))
print('O(1) content-addressed hit  :', us(t_hit),  '  <- well below 1 ms')
print('numpy on WASM CPU           :', us(t_cpu),  '  <- sub-ms')
print('cache:', holo_gpu.cache_stats())

## 3 · Keep it on the GPU — chain kernels, read back once
`readback=False` returns a GPU handle you feed into the next kernel. A chain pays the readback floor **once**, not per step.

In [ ]:
a = (np.arange(1<<18, dtype=np.float32) % 1000)
async def chained(n):
    h = await holo_gpu.run(wgsl, a, readback=False)
    for _ in range(n-1): h = await holo_gpu.run(wgsl, h, readback=False)
    return await holo_gpu.read(h)
async def separate(n):
    r = a
    for _ in range(n): r = (await holo_gpu.run(wgsl, r, cache=False)).data
    return r
t=time.perf_counter(); await chained(8);  tc=time.perf_counter()-t
t=time.perf_counter(); await separate(8); ts=time.perf_counter()-t
print(f'8 GPU kernels, one readback : {tc*1e3:6.2f} ms')
print(f'8 GPU kernels, 8 readbacks  : {ts*1e3:6.2f} ms')
print(f'on-GPU chaining speedup     : {ts/tc:4.1f}x')

## 4 · A non–von-Neumann primitive: balanced ternary on the GPU

In [ ]:
ternary = r"""
@group(0) @binding(0) var<storage, read_write> data: array<f32>;
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let i = gid.x;
    if (i < arrayLength(&data)) { let v = data[i]; data[i] = select(select(0.0, 1.0, v > 0.5), -1.0, v < -0.5); }
}"""
rng = np.random.default_rng(0)
v = rng.standard_normal(20).astype(np.float32)
t = await holo_gpu.run(ternary, v)
print('values:', np.round(v, 2))
print('trits :', np.asarray(t))

## 5 · Interactive — widgets + Plotly (feature-complete, offline)

In [ ]:
import ipywidgets as W
from IPython.display import display
slider = W.IntSlider(value=3, min=1, max=12, description='power')
out = W.Output()
def on_change(_=None):
    out.clear_output(wait=True)
    with out: print(f'powers 1..{slider.value} cubed =', [k**3 for k in range(1, slider.value+1)])
slider.observe(on_change, names='value'); on_change()
display(slider, out)

In [ ]:
import plotly.express as px
xs = np.linspace(0, 6.28, 200)
px.line(x=xs, y=np.sin(xs)*np.exp(-xs/6), title='Plotly, rendered in-browser')

### How it works
- `holo_gpu.address(arr)` → κ once; `run(kernel, addressed)` repeats are O(1) (microseconds).
- `run(..., readback=False)` keeps data on the GPU; chain, then `read()` once.
- Device, compiled pipelines (by κ), and buffers are pooled and reused.
- Build geometric-algebra, quantum state-vector, or cellular-automata kernels the same way.
- Everything ran on your hardware, served from content-addressed bytes — no server.